In [ ]:
import numpy as np 
import pandas as pd 
import os
print(os.listdir("../input/img_align_celeba"))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import make_grid
import torchvision.utils as vutils
import matplotlib.animation as animation
from IPython.display import HTML

import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import pandas as pd
import copy
import time
import cv2 as cv
from tqdm import tqdm_notebook as tqdm
import matplotlib.image as mpimg

In [ ]:
datapath = "../input/img_align_celeba/img_align_celeba/"
images_path = os.listdir("../input/img_align_celeba/img_align_celeba/")
print(len(images_path))
images_path = images_path[:]
valid_ratio = 0.8

In [ ]:
import torchvision.transforms.functional as TF  
class ImageData(Dataset):
    def __init__(self, is_train=True):
        self.is_train = is_train
        self.transform = transforms.Compose([transforms.ToTensor()])
        self.train_index = int(valid_ratio * len(images_path))
        self.crop = transforms.Resize((224, 224))  
    def __len__(self):
        if self.is_train:
            return self.train_index
        else:
            return len(images_path) - self.train_index - 1

    def __getitem__(self, index):
        if not self.is_train:
            index = self.train_index + index
        img = mpimg.imread(datapath + str(images_path[index]))
        img = self.crop(TF.to_pil_image(img))
        img = self.transform(img)
        img = (img - 0.5) / 0.5
        return img


In [ ]:
!pip install torchinfo

In [ ]:
batch_size=20
dataset = ImageData()
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
device = 'cuda'

In [ ]:
a = next(iter(dataloader))
print(a[0].shape)
img = a[15]
img = img *0.5 + 0.5
plt.imshow(img.permute(1,2,0))

In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

In [ ]:
IMG_WIDTH = 224
IMG_HEIGHT = 224
latent_size = 200

In [ ]:
!pip install linformer

In [ ]:
class GumbelQuantize(nn.Module):
    def __init__(self, num_hiddens, n_embed, embedding_dim, straight_through=False, kld_scale=5e-5, init_tau=1.0, min_tau=0.1, anneal_rate=0.00005):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.n_embed = n_embed
        self.straight_through = straight_through
        self.temperature = init_tau
        self.min_tau = min_tau
        self.anneal_rate = anneal_rate
        self.kld_scale = kld_scale
        self.proj = nn.Conv2d(64, n_embed, 1)
        self.embed = nn.Embedding(n_embed, embedding_dim)
        nn.init.orthogonal_(self.embed.weight)
    
    def forward(self, z):
        logits = self.proj(z)
        self.temperature = max(self.min_tau, self.temperature * (1 - self.anneal_rate))
        hard = self.straight_through if self.training else True
        soft_one_hot = F.gumbel_softmax(logits, tau=self.temperature, dim=1, hard=hard)
        z_q = torch.matmul(soft_one_hot.permute(0, 2, 3, 1), self.embed.weight).permute(0, 3, 1, 2)
        qy = F.softmax(logits, dim=1)
        diff = self.kld_scale * torch.sum(qy * torch.log(qy * self.n_embed + 1e-10), dim=1).mean()
        indices = soft_one_hot.argmax(dim=1)
        return z_q, diff


In [ ]:
import torch
import torch.nn as nn
from linformer import Linformer

class Encoder(nn.Module):
    def __init__(self, num_channels_in_encoder=8):  
        super(Encoder, self).__init__()
        self.e_conv_1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=(3, 3), stride=(2, 2), padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), stride=(1, 1), padding=1),  
            nn.ReLU()
        )
        self.e_conv_2 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), stride=(2, 2), padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), stride=(1, 1), padding=1),  
            nn.ReLU()
        )
        self.e_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.linformer_block = None
        self.e_conv_3 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), stride=(2, 2), padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), stride=(1, 1), padding=1), 
            nn.ReLU()
        )
        self.gumbel_quant = GumbelQuantize(num_hiddens=64, n_embed=64 ,embedding_dim=8)
    def forward(self, x):
        ec1 = self.e_conv_1(x)  
        ec2 = self.e_conv_2(ec1)  
        eblock1 = self.e_block_1(ec2) + ec2 
        batch_size, channels, height, width = eblock1.shape
        seq_len = height * width  
        if self.linformer_block is None:
            self.linformer_block = Linformer(
                dim=64, seq_len=seq_len, depth=1, heads=4, k=64
            ).to(x.device)  
        eblock1_flat = eblock1.view(batch_size, seq_len, channels)  
        linform = self.linformer_block(eblock1_flat) 
        linform_reshaped = linform.view(batch_size, channels, height, width)
        ec3 = self.e_conv_3(linform_reshaped)  
        z_q, diff = self.gumbel_quant(ec3)
        return z_q, diff
        
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
netE = Encoder(num_channels_in_encoder=8).to(device)
inp = torch.randn(IMG_WIDTH * IMG_HEIGHT * 3 * 100).view((-1, 3, IMG_HEIGHT, IMG_WIDTH)).to(device)
output = netE(inp)[0]
print(output.shape)  
print('The Compression Ratio is :  ' + str((output.shape[1] * output.shape[2] * output.shape[3]) / (IMG_WIDTH * IMG_HEIGHT * 3) * 100))


In [ ]:
from torchinfo import summary
summary(netE)

In [ ]:
import torch
import torch.nn as nn

class WGAN_Generator(nn.Module):
    def __init__(self):
        super(WGAN_Generator, self).__init__()
        self.d_up_conv_1 = nn.Sequential(
            nn.Conv2d(in_channels=8, out_channels=32, kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(in_channels=64, out_channels=64, kernel_size=(2, 2), stride=(2, 2)),  
            nn.ReLU(),
        )
        self.d_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )
        self.d_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.d_up_conv_2 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(in_channels=128, out_channels=128, kernel_size=(2, 2), stride=(2, 2)),  
            nn.ReLU(),
            nn.ConvTranspose2d(in_channels=128, out_channels=64, kernel_size=(2, 2), stride=(2, 2)),
            nn.ReLU(),
        )
        self.d_final_conv = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=32, kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=16, kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=16, out_channels=3, kernel_size=(3, 3), stride=(1, 1), padding=1),
            nn.Tanh()  
        )

    def forward(self, x):
        uc1 = self.d_up_conv_1(x)  
        dblock1 = self.d_block_1(uc1) + uc1  
        dblock2 = self.d_block_2(dblock1) + dblock1  
        uc2 = self.d_up_conv_2(dblock2)  
        dec = self.d_final_conv(uc2)  
        return dec


In [ ]:
netG = WGAN_Generator().to(device)
netG.apply(weights_init)
inp = torch.randn(100*num_channels_in_encoder*28*28).view((-1,num_channels_in_encoder,28,28)).to(device)
output = netG(inp)
print(output.shape)


In [ ]:
import torch
import torch.nn as nn

class WGAN_Discriminator(nn.Module): #critic
    def __init__(self):
        super(WGAN_Discriminator, self).__init__()
        self.latent_layer1 = nn.Sequential(
            nn.ConvTranspose2d(8, 16, (3,3), stride=2, padding=1, output_padding=1),  
            nn.ReLU(inplace=True),
        )
        self.latent_layer2 = nn.Sequential(
            nn.ConvTranspose2d(16, 32, (3,3), stride=2, padding=1, output_padding=1),  
            nn.ReLU(inplace=True),
        )
        self.latent_layer3 = nn.Sequential(
            nn.ConvTranspose2d(32, 64, (3,3), stride=2, padding=1, output_padding=1),  
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 64, (3,3), stride=1, padding=1, output_padding=0), 
            nn.ReLU(inplace=True),
        )
        self.latent_layer4 = nn.Sequential(
            nn.ConvTranspose2d(64, 3, (3,3), stride=1, padding=1, output_padding=0),  
            nn.ReLU(inplace=True),
        )
    
        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channels=6, out_channels=64, kernel_size=3, stride=1, padding=1),  
            nn.ReLU(inplace=True),
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=4, stride=2, padding=1),  
            nn.ReLU(inplace=True),
        )
        self.layer3 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=32, kernel_size=4, stride=2, padding=1), 
            nn.ReLU(inplace=True),
        )
        self.layer4 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=16, kernel_size=4, stride=2, padding=1),  
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=16, out_channels=2, kernel_size=3, stride=1, padding=1),  
            nn.ReLU(inplace=True),
        )
        self.fc1 = nn.Linear(2*28*28, 256)
        self.fc2 = nn.Linear(256, 100)  
        self.fc3 = nn.Linear(100,1)
    
    def forward(self, inp):
        encoded = inp['encoded'].to(device)  
        x = inp['img'].to(device)  
        y = self.latent_layer1(encoded)
        y = self.latent_layer2(y)
        y = self.latent_layer3(y)
        y = self.latent_layer4(y) 
        x = torch.cat((x, y), 1)  
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = x.reshape((x.shape[0], -1))  
        x = self.fc1(x)  
        x = self.fc2(x)   
        x = self.fc3(x)   
        return x

In [ ]:
netD = WGAN_Discriminator().to(device)
netD.apply(weights_init)
inp_x = {}
inp_x['img']=torch.randn(IMG_WIDTH*IMG_HEIGHT*3 * 100).view((-1,3,IMG_HEIGHT,IMG_WIDTH))
print("<--debug-->")
print(IMG_HEIGHT)
inp_x['encoded'] = torch.randn(100*num_channels_in_encoder*28*28).view((-1,num_channels_in_encoder,28,28))
output = netD(inp_x)
output.shape

In [ ]:
lr = 0.000009
lrd = 0.000009
criterion = nn.BCELoss()
msecriterion = nn.MSELoss()
l1criterion = nn.L1Loss()
real_label = 1
fake_label = 0
optimizerD = optim.Adam(netD.parameters(), lr=lrd, betas=(0.3, 0.5))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(0.45, 0.8))
optimizerE = optim.Adam(netE.parameters(), lr=lr, betas=(0.45, 0.8))

In [ ]:
valid_dataset = ImageData(is_train=False)
num_images_to_show = 1
valid_dataloader = DataLoader(valid_dataset, batch_size=num_images_to_show, shuffle=True)
valid_batch = next(iter(valid_dataloader)).to(device)

In [ ]:
def gradient_penalty(netD, real_data, fake_data, encoded_data):
    batch_size = real_data.size(0)
    epsilon = torch.rand(batch_size, 1, 1, 1, device=real_data.device)  # Mixing coefficient
    interpolates = epsilon * real_data + (1 - epsilon) * fake_data
    interpolates.requires_grad_(True)

    inp_x_interp = {'img': interpolates, 'encoded': encoded_data}
    d_interpolates = netD(inp_x_interp)
    grad_outputs = torch.ones_like(d_interpolates, device=real_data.device)

    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=grad_outputs,
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]
    gradients = gradients.view(batch_size, -1)
    gradient_norm = gradients.norm(2, dim=1)
    penalty = ((gradient_norm - 1) ** 2).mean()
    return penalty


In [ ]:
import torch
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
def calculate_psnr(img1, img2):
    mse = F.mse_loss(img1, img2)
    if mse == 0:
        return float("inf")
    psnr = 20 * torch.log10(1.0 / torch.sqrt(mse))
    return psnr
G_losses = []
D_losses = []
E_losses = []
psnr_values = []  
iters = 0
num_epochs = 2
num_images_to_show = 4  
print("Starting Training Loop...")

for epoch in range(num_epochs):
    epoch_psnr_values = []  
    for i, (images) in enumerate(dataloader, 0):
        netG.train()
        netD.train()
        netE.train()
        netD.zero_grad()
        images = images.to(device)
        encoded_output,kld = netE(images)
        fake_images = netG(encoded_output)
        inp_x = {}
        inp_x['img'] = images
        inp_x['encoded'] = encoded_output
        output_real = netD(inp_x).view(-1)
        inp_x_fake = {}
        inp_x_fake['img'] = fake_images
        inp_x_fake['encoded'] = encoded_output
        output_fake = netD(inp_x_fake).view(-1)
        lambda_gp = 1 
        gp = gradient_penalty(netD, images, fake_images , encoded_output)
        errD = -output_real.mean() + output_fake.mean() + lambda_gp * gp
        errD.backward(retain_graph=True)
        optimizerD.step()
        D_x = output_real.mean().item()
        D_G_z1 = output_fake.mean().item()
        netG.zero_grad()
        inp_x_fake = {}
        inp_x_fake['img'] = fake_images
        inp_x_fake['encoded'] = encoded_output

        output_fake = netD(inp_x_fake).view(-1)
        errG = -output_fake.mean() + 4 * l1criterion(images, fake_images)
        errG.backward(retain_graph=True)
        optimizerG.step()

        D_G_z2 = output_fake.mean().item()
        netE.zero_grad()
        inp_x_fake = {}
        inp_x_fake['img'] = fake_images
        inp_x_fake['encoded'] = encoded_output
        output_fake = netD(inp_x_fake).view(-1)
        errE = -output_fake.mean() + 4 * l1criterion(images, fake_images) + 1.5*kld
        errE.backward(retain_graph=True)
        optimizerE.step()
        if i % 50 == 0:
            print('[%d/%d][%d/%d]\tLoss_D: %.4f\tLoss_G: %.4f\tLoss_E: %.4f\tD(x): %.4f\tD(G(z)): %.4f / %.4f'
                  % (epoch, num_epochs, i, len(dataloader),
                     errD.item(), errG.item(), errE.item(), D_x, D_G_z1, D_G_z2))
        G_losses.append(errG.item())
        D_losses.append(errD.item())
        E_losses.append(errE.item())
        psnr_value = calculate_psnr(fake_images, images)
        epoch_psnr_values.append(psnr_value.item())  
        del images
        del inp_x_fake
        del inp_x
        del output_real
        del output_fake
        torch.cuda.empty_cache()
        iters += 1
        if i % 100 == 0:
            netE.eval()
            netG.eval()
            encoded_img,_ = netE(valid_batch)
            reconstructed_img = netG(encoded_img)
            f, axarr = plt.subplots(num_images_to_show, 2, squeeze=False)
            f.set_figheight(20)
            f.set_figwidth(20)
            batch_size = valid_batch.size(0)
            images_to_display = min(num_images_to_show, batch_size)
            for j in range(images_to_display):
                valid_img = (valid_batch[j].cpu().detach().permute(1, 2, 0) * 0.5) + 0.5
                rec_img = (reconstructed_img[j].cpu().detach().permute(1, 2, 0) * 0.5) + 0.5
                axarr[j, 0].imshow(valid_img)
                axarr[j, 1].imshow(rec_img)
                axarr[j, 0].set_title("Original")
                axarr[j, 1].set_title("Reconstructed")
            for j in range(images_to_display, num_images_to_show):
                f.delaxes(axarr[j, 0])
                f.delaxes(axarr[j, 1])
            plt.show()
            plt.close(f)  
            avg_psnr = np.mean(epoch_psnr_values)
            print(f'Epoch [{epoch + 1}/{num_epochs}], Average PSNR: {avg_psnr:.2f} dB')
    torch.save(netG.state_dict(), f'netG_epoch_{epoch+1}.pth')
    torch.save(netE.state_dict(), f'netE_epoch_{epoch+1}.pth')
    torch.save(netD.state_dict(), f'netD_epoch_{epoch+1}.pth')
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 3, 1)
    plt.plot(G_losses, label='Generator Loss')
    plt.title('Generator Loss')
    plt.xlabel('Iterations')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig('generator_loss.jpeg', format='jpeg')
    plt.subplot(1, 3, 2)
    plt.plot(D_losses, label='Discriminator Loss')
    plt.title('Discriminator Loss')
    plt.xlabel('Iterations')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig('critic_loss.jpeg', format='jpeg')
    plt.subplot(1, 3, 3)
    plt.plot(E_losses, label='Encoder Loss')
    plt.title('Encoder Loss')
    plt.xlabel('Iterations')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig('encoder_loss.jpeg', format='jpeg')
    plt.tight_layout()
    plt.show()